# Deelmobiliteit
This document examplifies how to take an existing norm and translate it into an eFLINT specification that can be used to enforce it.

This example focusses on the entire process, not just the eFLINT side of it. As such, eFLINT won't be introduced until a bit later in this document.

## The use-case
Let's start by introducing the use-case.

In this scenario, a municipality (`municipality`) has different provides of shared mobility services that all offer their services through their own applications. Imagine that there are three: Amy's Bicycles TO GO (`amy`), Bobcycles (`bob`) and The Dan Bicycle Network, Inc. (`dan`). We also have some users, Eve (`eve`) and Fey (`fey`) who might be interested in renting shared bicycles.

The goal of the municipality is as follows: instead of all three provides exclusively offering their own bicycles, it should be possible to rent other companies' bicycles through other companies' apps. For example, when Fey logs in to the app of Amy's Bicycles TO GO, she must not only see the offer of them but also that of Bobcycles and The Dan Bicycle Network, Inc. In the background, `amy`'s services must negotiate with `bob`'s and `dan`'s to rent the bicycle on Fey's behalf.

## The norms
Let's define a few norms that will cater to this use-case in the form of an Afsprakenstelsel that defines how all parties must behave.

First, we shall formalise the wish of the manicupality that all mobility services must offer each other's bicycles:
> **Article 1** Every _mobility service provider_ must _present_ the _offers_ of other mobility service providers to their _users_ in addition to their own.

Next, let's define that customer information is sensitive; i.e.:
> **Article 2** _Identifying information_ of users cannot be _shared_ by a mobility service providers to other mobility service providers.

## The model
With these norms in mind, it is time to build a _model_ of the scenario describes above. This is important, because the reasoner cannot physically observe or interact with the system; so instead, we need to tell it what the system consists of, how it behaves and what we want it to do. This is all captured by the model.

Specifically, most models tends to have four layers of abstraction:
1. We define an _knowledge representation_, which describes what the system consists of;
2. We define _process model_, which describes how the system (can) behave;
3. We define a _normative classification_, which classify possible behaviours as either good or bad, communicating what we want the system to do; and
4. We define a _scenario specification_, which populates specific instances of the system that we model.

This is usually an iterative process, where the knowledge representation is influenced by how violations are described and vice versa. However, in this notebook, we'll do 1-3 in order, but interleave step 4 with the others to be able to be a little more concrete. Where the scenario is defined is actually relatively irrelevant, except that it is usually saved last for practical purposes.

## The knowledge representation
First, we need to describe what the scenario above consists of by virtue of creating a representation of the model's knowledge. You can think of this as "defining the universe": this step lays the groundwork for what is possible to express for the process model and the normative classification.

For our example, we can find the universe of all possible things by studying the norms and discovering what they concern themselves with. We can see that there are two types of agents in the system: _mobility service providers_ and _users_. Further, it seems like there are also two things called _identifying information_ and _offers_, which have specific relations with either of the two agents: identifying information describes a particular user and is shared by mobility service providers; and similarly, offers are made by mobility service providers, potentially on behalf of other providers, and are accepted by users.

If we want to formalize the above in a form that the eFLINT reasoner understands, we have to formalize them as _facts_. Specifically, we can phrase the knowledge representation as a set of _claims_ that we are able to make in our model. Concrete things like:
```plain
Fey is a user.
Amy's Bicycles TO GO is a mobility service provider.
EVE_DATA identifies Eve.
...
```
Again, remember that this sets up the universe of our model. If something cannot be claimed to hold true, then we cannot possible describe it and therefore we cannot include it in our model.

To be useful for eFLINT, however, we don't list all possible claims, but rather the structure of all possible claims (i.e., types). Instead of the above, we write down something like:
```plain
X is a user.
X is a mobility service provider.
X identifies Y.
...
```
where `X` and `Y` are variables that act as placeholders for concrete things that we want to make claims about.

This way, we can now completely transform the textual knowledge representation above into a universe of possible claims that we can make:
```plain
X is an entity.
Some entity X is a user.
Some entity X is a mobility service provider.
X is identifying data.
X is an offer.
Some identifying data X identifies some user Y.
Some identifying data X is shared with entity Y.
Some offer X is offered by mobility service provider Y on behalf of mobility service provider Z.
Some offer X is accepted by user Y.
```
> Once more: the set above describes the set of **all possible things that we might consider**. This does not yet take norms into account; this model finds it perfectly acceptable that two mobility service share identifying data with each other. While not permitted by the norms, the reality is that service can physically do this action.

Reasoning about claims like these are true or false is exactly what we do in eFLINT. At all times, eFLINT keeps track of a _knowledge base_ that logs if a claim is known to be true or otherwise assumed to be false (closed world assumption). Then, in the specification, we can define shapes of facts as we did with the claims:

In [1]:
// The four entities/things
Fact entity Identified by String.      // String plays the role here of a general "thing"; everything is, at the end of the day, a thing.
Fact user Identified by entity.
Fact provider Identified by entity.    // Mobility service provider but more abbreve
Fact identifying-data Identified by String.
Fact offer Identified by String.

// Their relationships
Fact identifies Identified by identifying-data * user.
Fact shared-with Identified by identifying-data * entity.
Fact offered-by-on-behalf-of Identified by offer * provider1 * provider2.
Fact accepted-by Identified by offer * user.

Later, we can make concrete claims that fit the shapes of the knowledge representation above:

In [2]:
// Let's define some entities!
+entity("Amy's Bicycles TO GO").               // Define that Amy is an entity
+provider(entity("Amy's Bicycles TO GO")).     // Further specify that Amy is a provider.

+entity("Bobcycles").
+provider(entity("Bobcycles")).

+entity("The Dan Bicycle Network, Inc.").
+provider(entity("The Dan Bicycle Network, Inc.")).

+entity("Eve").
+user(entity("Eve")).
+entity("Fey").
+user(entity("Fey")).

// And let's define some data while at it
+identifying-data("EVE_DATA").
+identifies(identifying-data("EVE_DATA"), user(entity("Eve"))).

+identifying-data("FEY_DATA").
+identifies(identifying-data("FEY_DATA"), user(entity("Fey"))).

+entity("Amy's Bicycles TO GO")
+entity("Bobcycles")
+identifying-data("EVE_DATA")
+entity("Eve")
+identifying-data("FEY_DATA")
+entity("Fey")
+entity("The Dan Bicycle Network, Inc.")
+provider(entity("Amy's Bicycles TO GO"))
+provider(entity("Bobcycles"))
+identifies(identifying-data("EVE_DATA"),user(entity("Eve")))
+user(entity("Eve"))
+identifies(identifying-data("FEY_DATA"),user(entity("Fey")))
+user(entity("Fey"))
+provider(entity("The Dan Bicycle Network, Inc."))


> ### Knowledge representation... or input?
> The first step -building the knowledge representation- is a very important step. Not only does it define what can be expressed in the rest of the model, it also defines what needs to be given as input to your model; and this means that it sets the horizon for what must be _qualified_.
> 
> An important part of any legal reasoning process is the step of qualification: translating real-world events and objects to their legal interpretations. This is the same that happens in a digital twin through sensors or monitoring in social systems: it is the act of recognizing something in the real world as something in a model or virtual world.
> 
> In eFLINT, this is formalised explicitly as deciding what input to give to the model. If a user accepts an offer in real life, I qualify it as a legal offer by setting the appropriate input in the model. Therefore, the input that the specification requires determines how granular the qualifications need to be: a spec that has `shake-hands` as input is much easier to qualify than a spec that has `meets` as input. Which of the two to select is completely dependent on the use-case.

## The process model
Now that we have an idea of _what_ we can express, it is time to define how the system _behaves_. This is done by defining the process model.

In eFLINT, this is most literally formalized by virtue of defining _actions_. An action formalises a right of some thing to change the world in a particular way. The effect of an action is defined as making new claims true, or old ones false, as dictated by its `Creates` or `Terminates`-clause. For example, an action `eat-apple` would make the fact that the eaten apple exists false, and might make the fact that that person is satisfied true.

Looking back at the norms part of this use-case, we observe three actions that can occur: any entity can _share_ identifying data with another entity; a mobility service provider can _offer_ a new service; and a user can _accept_ an existing offer. The effects of the actions are defined in terms of the knowledge representation: e.g., when en entity `share`s something, a claim that says it is `shared-with` is created to model the change.

For example, we can define the fact `share` as follows:

In [3]:
// Action that shares data between entities
Act share
    // Every action has an actor, which some thing (usually an entity) that performs the action.
    Actor entity1
    // Some actions also have recipients, which are things that receive the actions. But this is not necessary (see below).
    Recipient entity2
    // Some actions have other things they relate to. In this case, we encode here the data that is being shared.
    Related to identifying-data
    // When `entity1` takes this action, they create the claim that the data has been shared with `entity2`
    Creates shared-with(identifying-data, entity2)
    // This specifies that, basically, the action can always be taken. More about it below!
    Holds when True.

+share(entity("Amy's Bicycles TO GO"),entity("Amy's Bicycles TO GO"),identifying-data("EVE_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Amy's Bicycles TO GO"),identifying-data("FEY_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("EVE_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("FEY_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Eve"),identifying-data("EVE_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Eve"),identifying-data("FEY_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Fey"),identifying-data("EVE_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("Fey"),identifying-data("FEY_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("The Dan Bicycle Network, Inc."),identifying-data("EVE_DATA"))
+share(entity("Amy's Bicycles TO GO"),entity("The Dan Bicycle Network, Inc."),identifying-data("FEY_DATA"))
+share(entity("Bobcycles"),entity("Amy's Bicycles TO GO"),identifying-data("EVE_DA

Observe that this defines A LOT of actions! Like Facts, any `Act`-definition merely defines the _shape_ of an action. When we create an Act like this, eFLINT will automatically create a concrete one for every possible combination of entities and identifying data. What this means precisely is discussed below.

We might use one of these actions as follows:

In [4]:
share(entity("Amy's Bicycles TO GO"), entity("Bobcycles"), identifying-data("EVE_DATA")).

+shared-with(identifying-data("EVE_DATA"),entity("Bobcycles"))


Executed transition: share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("EVE_DATA"))



As you can see, the action dutifully created the appropriate `shared-with`-claim.

Let's finish this step and define the other actions that entities can take:

In [5]:
// Action that creates an offer.
Act make-offer
    Actor provider
    Related to offer
    // When the `provider` takes this action, they create the claim that they have offered the offer on behalf of themselves.
    Creates offered-by-on-behalf-of(offer, provider, provider)
    Holds when True.

// Action that creates an offer on someone else's behalf.
Act make-offer-on-behalf-of
    Actor provider1
    Related to provider2, offer
    // When the `provider1` takes this action, they create the claim that they have offered the offer on behalf of `provider2`.
    Creates offered-by-on-behalf-of(offer, provider1, provider2)
    Holds when True.

// Action that causes a user to accept an offer.
Act accept-offer
    Actor user
    Recipient provider
    Related to offer
    // When the `user` takes this action, they create the claim that they have accepted the offer.
    Creates accepted-by(offer, user)
    Holds when True.

## Normative classification

The final step of building the model is defining which states of the knowledge base should be considered compliant and which should be considered violated.

In the Hohfeldian Framework (<https://compass.onlinelibrary.wiley.com/doi/pdf/10.1111/phc3.12342>), upon which eFLINT is based, there are two ways of doing so: we can add prohibitions to actions (i.e., specify when an action is illegal) and we can create duties that model an obligation of agents. Let's first look at restriction actions.

### Actions

Like Facts, an Action in eFLINT has a _truth value_ associated with it (i.e., Acts can be True or False). If an Act is True, we say that it is _enabled_; this simply means that we consider taking that action appropriate. By contrast, if it's False, we say it is _disabled_, which we interpret as that taking that action would be a violation of the norms.

Above, we had the weird `Holds when True`-clause. This simply tells eFLINT that, by default, the Act is enabled for _all_ possible combinations of entities and identifying data. This is fine, but not what we want to model; Article 2 states that taking this action is actually a violation if the sharing happens between mobility service providers. To model that, we can _extend_ the Act with a `Conditioned by`-clause, which acts as a filter to determine which actions are true and which aren't. Specifically, it allows us to formulate a condition that makes the act _legal_.

To formalise Article 2, we can thus attach a `Conditioned by` that will define all actions legal IF **not** both entities are mobility service providers:

In [6]:
Extend Act share
    Conditioned by Not(provider(entity1) && provider(entity2)).

~share(entity("Amy's Bicycles TO GO"),entity("Amy's Bicycles TO GO"),identifying-data("EVE_DATA"))
~share(entity("Amy's Bicycles TO GO"),entity("Amy's Bicycles TO GO"),identifying-data("FEY_DATA"))
~share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("EVE_DATA"))
~share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("FEY_DATA"))
~share(entity("Amy's Bicycles TO GO"),entity("The Dan Bicycle Network, Inc."),identifying-data("EVE_DATA"))
~share(entity("Amy's Bicycles TO GO"),entity("The Dan Bicycle Network, Inc."),identifying-data("FEY_DATA"))
~share(entity("Bobcycles"),entity("Amy's Bicycles TO GO"),identifying-data("EVE_DATA"))
~share(entity("Bobcycles"),entity("Amy's Bicycles TO GO"),identifying-data("FEY_DATA"))
~share(entity("Bobcycles"),entity("Bobcycles"),identifying-data("EVE_DATA"))
~share(entity("Bobcycles"),entity("Bobcycles"),identifying-data("FEY_DATA"))
~share(entity("Bobcycles"),entity("The Dan Bicycle Network, Inc."),identifying-d

Note that, because the norm is phrased negatively (a prohibition), we actually formalise the opposite of the norm: "you ONLY have the right to share if you're NOT sharing between providers".

Further note that the moment we attach the claim, eFLINT retracts some actions that used to be legal but aren't anymore. If we formalised our norm right, all actions that concern sharing identifyable information between two entities that are mobility service providers are now violating.

Now, if we try to perform this action, eFLINT tells us:

In [7]:
share(entity("Amy's Bicycles TO GO"), entity("Bobcycles"), identifying-data("EVE_DATA")).

Executed transition: share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("EVE_DATA"))

Disabled action: share(entity("Amy's Bicycles TO GO"),entity("Bobcycles"),identifying-data("EVE_DATA"))


As expected, eFLINT tells us that the action was disabled! We successfully introduced our first norm to the model.

### Duties

The other form of modelling norms is by giving things _obligations_. An obligation encodes that they must do something or enact some change before a certain condition is met: a well-known example of an obligation is paying before you exit a shop.

eFLINT models this explicitly through the special `Duty`-construct. Duties always have a `Holder`, which the thing that has the obligation, and a `Claimant`, which is the thing that holds the holder responsible for completing the duty. Like Facts and Acts, Duties have a truth value associated with them that encodes whether a duty _holds_; in the example of paying, there would be a condition that you only have to pay when you have picked up an item. Then, they also have a secondary truth value computed from a `Violated when`-clause: this encodes the condition that would cause you to violate the duty if you still have it. For example, this would be that you're outside the shop without having paid.

We can formalize article 1 in this way. We will interpret it as encoding a Duty that holds when any mobility service provider offer something, all the others must offer that on behalf of the first. The violation condition is a bit vague, so we'll interpret it to mean that the duty is violated when the offer has been accepted at any location but the provider hasn't offered it yet.

The eFLINT:

In [8]:
Duty must-offer-on-behalf-of
    Holder provider1
    Claimant provider2
    Related to offer
    // The Duty EXISTS when provider2 has made the offer on behalf of itself, but provider 1 hasn't offered it yet
    Holds when (
        offered-by-on-behalf-of(offer, provider2, provider2)
        && Not(offered-by-on-behalf-of(offer, provider1, provider2))
    )
    // The Duty is VIOLATED if the offer is accepted ANYWHERE while the Duty holds
    Violated when (Exists user : accepted-by(offer, user)).

Now, let us consider that an offer is about to be made by Amy's Bicycles TO GO:

In [9]:
// We define the offer exists
+offer("Bicycle A").

+offer("Bicycle A")
+make-offer(provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))
+make-offer-on-behalf-of(provider(entity("Amy's Bicycles TO GO")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))
+make-offer-on-behalf-of(provider(entity("Amy's Bicycles TO GO")),provider(entity("Bobcycles")),offer("Bicycle A"))
+make-offer-on-behalf-of(provider(entity("Amy's Bicycles TO GO")),provider(entity("The Dan Bicycle Network, Inc.")),offer("Bicycle A"))
+make-offer(provider(entity("Bobcycles")),offer("Bicycle A"))
+make-offer-on-behalf-of(provider(entity("Bobcycles")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))
+make-offer-on-behalf-of(provider(entity("Bobcycles")),provider(entity("Bobcycles")),offer("Bicycle A"))
+make-offer-on-behalf-of(provider(entity("Bobcycles")),provider(entity("The Dan Bicycle Network, Inc.")),offer("Bicycle A"))
+accept-offer(user(entity("Eve")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))
+accept-offer(user(entity

This enables a lot of actions, because all of these actions relate to offers and none existed before.

But the interesting thing happens now: Amy actually makes the offer:

In [10]:
// It is offered.
make-offer(provider("Amy's Bicycles TO GO"), offer("Bicycle A")).

+offered-by-on-behalf-of(offer("Bicycle A"),provider(entity("Amy's Bicycles TO GO")),provider(entity("Amy's Bicycles TO GO")))
+must-offer-on-behalf-of(provider(entity("Bobcycles")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))
+must-offer-on-behalf-of(provider(entity("The Dan Bicycle Network, Inc.")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))


Executed transition: make-offer(provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))



The second and third line tell us that a Duty has become enabled: both Bobcycles and The Dan Bicycle Network, Inc. must now both make the same offer on behalf of Amy's Bicycles TO GO.

Let's say that Bob does this:

In [11]:
make-offer-on-behalf-of(provider("Bobcycles"), provider("Amy's Bicycles TO GO"), offer("Bicycle A")).

~must-offer-on-behalf-of(provider(entity("Bobcycles")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))
+offered-by-on-behalf-of(offer("Bicycle A"),provider(entity("Bobcycles")),provider(entity("Amy's Bicycles TO GO")))


Executed transition: make-offer-on-behalf-of(provider(entity("Bobcycles")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))



the duty of Bobcycles to offer on Amy's behalf is now complete, and he can rest easy.

In [12]:
accept-offer(user("Eve"), provider("Amy's Bicycles TO GO"), offer("Bicycle A")).

+accepted-by(offer("Bicycle A"),user(entity("Eve")))


Executed transition: accept-offer(user(entity("Eve")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))

Violated duty!: must-offer-on-behalf-of(provider(entity("The Dan Bicycle Network, Inc.")),provider(entity("Amy's Bicycles TO GO")),offer("Bicycle A"))


But lo! Eve comes along and accepts Amy's Bicycles TO GO's offer to ride Bicycle A. Bobcycles is in the clear, because they have actually made the offer on behalf of Amy's; but The Dan Bicycle Network, Inc. hasn't, which has appropriately triggered a Duty which tells us that they have violated our norm.

## Qualification

So far so good. However, all of this is just observing whether a model enters a violating state and reporting some violations; none of this relates anything to the real world yet.

This is why the act of _qualificiation_ is so important. The system that is being modeled must be able to accurately obtain the information needed to feed the model, and this must be accurately given to the eFLINT reasoner. If the reasoner is not told of Bobcycles offering on Amy's Bicycles TO GO's behalf in the previous scenario, a violation would have been incorrectly triggered, blaming Bobcycles even though they did everything well. As such, the business of finding information in the system and orchestrating that is non-trivial.

Unfortunately, as observed, is the information required highly model-specific. This means that a slightly different model, which defines a different relation in its knowledge representation, might require vastly different input to be given to it by the system. This makes policies relatively tightly coupled with the system they are interfacing with; not every system is able to support any type of policy. Put differently, the degree of automation of a system's norms depends on the quality of the qualification.

On the other end, the system itself must be set up in such a way that the violations are being processed. This is either a human, who has the power to punish violating entities, or a machine that restricts (future) access, notifies a person, etc. Without proper actuation, the reasoner can reason all day but nothing will happen. And ensuring that its violations are accurately executed is non-trivial either.

## Norm compatability

While most norms can be formalised this way, not all norms are equally suitable for automation.

First, eFLINT is rather bad at doing floating-point arithmetic (as in, it doesn't support this). This means that norms who record a specific computation (e.g., scaling a value based on when in a year a party did something) need to be externalised outside of eFLINT.

Another thing is that norms may be very vague and still require interpretation. In our example, Article 1 completely unspecifies when the duty is violated; legal interpretation is required to understand what the violation condition of the associated duty is.

Finally, the more the norms discuss about conflict resolution, the more capable a system using the reasoner is at operating autonomously.

## Conclusion

This concludes the demo for translating norms into eFLINT specifications!

The takeaways are:
1. eFLINT specifications are models of the system they are checking;
2. We can usually define three layers: the _knowledge representation_ (i.e., input), the _process model_, the _normative classification_ and the _scenario specification_;
    1. The first deals with defining the possible states that the system may have;
    2. The process model defines how the system can evolve by things / agents taking actions;
    3. The normative classification defines which states of the system / evolutions of the state are good and which are bad; and
    4. The scenario specification feeds the reasoner with a specific state and evolution of that state to check against the norms.
3. The quality of the automation depends on the quality of the qualification; i.e., eFLINT specifications are very dependent on which- and how information is fed to them from the system they are checking; and
    - ...and this is very non-trivial.
4. A reasoner in isolation doesn't enforce anything; the output of the reasoner must also be appropriately handled by the wrapping system to be effective.
    - ...and this may influence how a specification is written.
5. Not all norms are equally automisable. The more concrete and detailled a norm is, the less interpretation is necessary when creating the specifications or enforcing them.